In [2]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/Karnataka_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)

features = ['NDVI', 'NDMI', 'NBR', 'BSI']
df = df[['latitude', 'longitude', 'year'] + features]
df = df.dropna()
df = df.sort_values(by=['latitude', 'longitude', 'year'])

# ===============================
# 3. Create Sequences (RAW DATA)
# ===============================
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/Karnataka_Deforestation_Predictions.csv',
    index=False
)

print("Saved: Karnataka_Deforestation_Predictions.csv")


Mounted at /content/drive
Raw input shape: (30000, 4, 4)

Label distribution:
No deforestation: 27827
Deforestation: 2173
Train shape: (24000, 4, 4)
Test shape : (6000, 4, 4)
Scaled train shape: (24000, 4, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.5340 - loss: 1.2552 - precision: 0.0825 - recall: 0.5135 - val_accuracy: 0.4698 - val_loss: 0.6648 - val_precision: 0.1130 - val_recall: 0.9218
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.5298 - loss: 1.0759 - precision: 0.1141 - recall: 0.8210 - val_accuracy: 0.5348 - val_loss: 0.5507 - val_precision: 0.1239 - val_recall: 0.8920
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.5060 - loss: 1.0528 - precision: 0.1147 - recall: 0.8836 - val_accuracy: 0.6345 - val_loss: 0.4514 - val_precision: 0.1453 - val_recall: 0.8276
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5994 - loss: 0.9749 - precision: 0.1364 - recall: 0.8471 - val_accuracy: 0.9422 - val_loss: 0.2813 - val_precision: 0.8188 - val_recall: 0.2598
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7077 - loss: 0.8817 - precision: 0.1751 - recall: 0.7896 - val_accuracy: 0.6053 - val_los

In [ ]:
!pip install geopandas folium shapely


In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from shapely.geometry import Point
from folium.plugins import MarkerCluster
from google.colab import drive

drive.mount('/content/drive')
df_def = pd.read_csv(
    '/content/drive/MyDrive/Karnataka_Deforestation_Predictions.csv'
)

df_deforest = df_def[df_def['deforestation'] == 1].copy()
print("Total deforestation samples:", len(df_deforest))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total deforestation samples: 683


In [4]:
districts = gpd.read_file(
    '/content/drive/MyDrive/Karnataka_District_Boundary.json'
)

districts = districts.to_crs(epsg=4326)
districts.head()


,cartodb_id,censuscode,dt_cen_cd,st_cen_cd,st_nm,district,geometry
0,115,570,16,29,Karnataka,Chikmagalur,"MULTIPOLYGON (((76.35411 13.5777, 76.35611 13...."
1,130,575,21,29,Karnataka,Dakshina Kannada,"MULTIPOLYGON (((75.20194 13.1814, 75.20992 13...."
2,138,567,13,29,Karnataka,Davanagere,"MULTIPOLYGON (((76.5083 14.56379, 76.50964 14...."
3,187,561,7,29,Karnataka,Gadag,"MULTIPOLYGON (((75.84568 15.83773, 75.87617 15..."
4,208,579,25,29,Karnataka,Gulbarga,"MULTIPOLYGON (((76.6922 17.68175, 76.69516 17...."


In [5]:
geometry = [Point(xy) for xy in zip(df_deforest.longitude, df_deforest.latitude)]
gdf_points = gpd.GeoDataFrame(df_deforest, geometry=geometry, crs="EPSG:4326")


In [6]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
df_def = pd.read_csv(
    '/content/drive/MyDrive/Karnataka_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))
df_cause = pd.read_csv(
    '/content/drive/MyDrive/Karnataka_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


Total samples: 6000
    latitude  longitude  deforestation
0  12.153757  75.703275              0
1  12.102553  75.767055              0
2  12.159147  75.657461              0
3  12.117824  75.671834              0
4  12.052247  75.804784              0
Total deforestation samples: 683
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  11.939059  75.886531              1    -0.322923   -0.242737    0.142379   
1  11.939059  75.890124              1    -0.292319   -0.236175    0.144617   
2  11.939958  75.882938              1    -0.317683   -0.252889    0.178680   
3  11.940856  75.880243              1    -0.337013   -0.268819    0.176103   
4  11.941754  75.883836              1    -0.320647   -0.259571    0.165498   

   NDMI_change        cause  
0    -0.107809  Urban/Other  
1    -0.113345  Urban/Other  
2    -0.150718  Urban/Other  
3    -0.141501  Agriculture  
4    -0.126970  Urban/Other  


In [7]:
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


    latitude  longitude  deforestation cause
0  12.162740  75.674529              1   NaN
1  11.975890  75.871260              1   NaN
2  12.072908  75.855988              1   NaN
3  11.975890  75.868565              1   NaN
4  12.057637  75.866768              1   NaN


In [8]:
geometry = [
    Point(xy) for xy in zip(df_deforest.longitude, df_deforest.latitude)
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)
districts = gpd.read_file(
    '/content/drive/MyDrive/Karnataka_District_Boundary.json'
)

districts = districts.to_crs("EPSG:4326")
print(districts.head())


   cartodb_id  censuscode  dt_cen_cd  st_cen_cd      st_nm          district  \
0         115         570         16         29  Karnataka       Chikmagalur   
1         130         575         21         29  Karnataka  Dakshina Kannada   
2         138         567         13         29  Karnataka        Davanagere   
3         187         561          7         29  Karnataka             Gadag   
4         208         579         25         29  Karnataka          Gulbarga   

                                            geometry  
0  MULTIPOLYGON (((76.35411 13.5777, 76.35611 13....  
1  MULTIPOLYGON (((75.20194 13.1814, 75.20992 13....  
2  MULTIPOLYGON (((76.5083 14.56379, 76.50964 14....  
3  MULTIPOLYGON (((75.84568 15.83773, 75.87617 15...  
4  MULTIPOLYGON (((76.6922 17.68175, 76.69516 17....  


In [9]:
gdf_joined = gpd.sjoin(
    gdf_points,
    districts,
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())


    latitude  longitude  deforestation cause                   geometry  \
0  12.162740  75.674529              1   NaN  POINT (75.67453 12.16274)   
1  11.975890  75.871260              1   NaN  POINT (75.87126 11.97589)   
2  12.072908  75.855988              1   NaN  POINT (75.85599 12.07291)   
3  11.975890  75.868565              1   NaN  POINT (75.86856 11.97589)   
4  12.057637  75.866768              1   NaN  POINT (75.86677 12.05764)   

   index_right  cartodb_id  censuscode  dt_cen_cd  st_cen_cd      st_nm  \
0         17.0       307.0       576.0       22.0       29.0  Karnataka   
1         17.0       307.0       576.0       22.0       29.0  Karnataka   
2         17.0       307.0       576.0       22.0       29.0  Karnataka   
3         17.0       307.0       576.0       22.0       29.0  Karnataka   
4         17.0       307.0       576.0       22.0       29.0  Karnataka   

  district  
0   Kodagu  
1   Kodagu  
2   Kodagu  
3   Kodagu  
4   Kodagu  


In [10]:
print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['district'].isna().sum())

print(gdf_joined['district'].value_counts())
district_summary = (
    gdf_joined
    .groupby('district')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)
district_summary.to_csv(
    '/content/drive/MyDrive/Karnataka_District_Wise_Deforestation.csv',
    index=False
)

print("Saved: Karnataka_District_Wise_Deforestation.csv")
cause_summary = (
    gdf_joined
    .groupby(['district', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())
cause_summary.to_csv(
    '/content/drive/MyDrive/Karnataka_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved: Karnataka_District_Wise_Deforestation_Causes.csv")


Total joined points: 683
Points without district: 33
district
Kodagu    650
Name: count, dtype: int64
  district  deforestation_points
0   Kodagu                   650
Sum of district-wise deforestation samples: 650
Saved: Karnataka_District_Wise_Deforestation.csv
  district    cause  count
0   Kodagu  Logging      9
Saved: Karnataka_District_Wise_Deforestation_Causes.csv


In [11]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster


In [12]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/Karnataka_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/Karnataka_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Function to compute thresholds
def get_thresholds(series):
    mean = series.mean()
    std = series.std()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# Compute thresholds for each index
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI thresholds:", ndvi_thresh)
print("NBR thresholds:", nbr_thresh)
print("BSI thresholds:", bsi_thresh)
print("NDMI thresholds:", ndmi_thresh)
df_merged = pd.merge(df_deforest, df_change[['latitude','longitude','NDVI_change','NBR_change','BSI_change','NDMI_change']],
                     on=['latitude','longitude'], how='left')
def classify_cause_adaptive(row):
    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']
    ndmi = row['NDMI_change']

    # 🔥 Fire: burn + moisture collapse
    if nbr < -0.30 and ndmi < -0.15:
        return "Fire"

    # 🪓 Logging: severe vegetation loss + strong soil exposure
    elif ndvi < -0.30 and bsi > 0.17:
        return "Logging"

    # 🌾 Agriculture: moderate loss + moderate soil exposure
    elif -0.30 <= ndvi < -0.20 and 0.13 <= bsi <= 0.17:
        return "Agriculture"

    else:
        return "Urban/Other"


# Apply classification
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)
df_merged.to_csv('/content/drive/MyDrive/Deforestation_Causes_Adaptive.csv', index=False)

# Summary
print("Deforestation Cause Summary (Adaptive Thresholds):")
print(df_merged['cause'].value_counts())
# Tamil Nadu roughly: latitude 8–13, longitude 76–80
# Karnataka roughly: latitude 11.5–18.5, longitude 74–78.5
m = folium.Map(location=[15.3, 76.4], zoom_start=7)

cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df_merged.iterrows():   # using df_merged (important)
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


NDVI thresholds: {'mean': np.float64(-0.3325616471321667), 'std': 0.03852233230463364, 'q1': np.float64(-0.3552701425), 'q3': np.float64(-0.3151820125), 'lower_2std': np.float64(-0.40960631174143397), 'upper_2std': np.float64(-0.2555169825228994)}
NBR thresholds: {'mean': np.float64(-0.24164152561473332), 'std': 0.03713549102738274, 'q1': np.float64(-0.26548166000000006), 'q3': np.float64(-0.22498631500000005), 'lower_2std': np.float64(-0.3159125076694988), 'upper_2std': np.float64(-0.16737054355996783)}
BSI thresholds: {'mean': np.float64(0.1449124511948783), 'std': 0.03611144057942998, 'q1': np.float64(0.1270652120902953), 'q3': np.float64(0.16960698402335964), 'lower_2std': np.float64(0.07268957003601835), 'upper_2std': np.float64(0.21713533235373828)}
NDMI thresholds: {'mean': np.float64(-0.12010152305205998), 'std': 0.032119967710382814, 'q1': np.float64(-0.1407969575), 'q3': np.float64(-0.10494005249999999), 'lower_2std': np.float64(-0.18434145847282563), 'upper_2std': np.float64

In [ ]:
m

In [14]:
m.save('/content/drive/MyDrive/Karnataka_Deforestation_Causes_Adaptive_Map.html')


In [17]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Karnataka District Boundaries
# =====================================================
ka = gpd.read_file('/content/drive/MyDrive/Karnataka_District_Boundary.json')

# CRS safety
if ka.crs is None:
    ka.set_crs(epsg=4326, inplace=True)

ka = ka.to_crs(epsg=4326)
ka["state"] = "Karnataka"
gdf_districts = ka.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/Deforestation_Causes_Adaptive.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("district")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("district")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("district")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="district", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="district", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="district", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Karnataka)
# =====================================================
m = folium.Map(location=[15.3, 76.4], zoom_start=6)

# =====================================================
# STEP 10: Dynamic Choropleth for Deforestation
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["district", "deforestation_samples"],
    key_on="feature.properties.district",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Karnataka)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "district",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Karnataka Afforestation Alert Popup
# =====================================================
state_total = gdf_districts["total_samples"].sum()
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['district']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Karnataka Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on plantation, controlled logging,<br>
and sustainable land management.
"""

folium.Marker(
    location=[15.3, 76.4],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/ka_forest.html")

print("✅ Karnataka map saved successfully with afforestation recommendation!")

✅ Karnataka map saved successfully with afforestation recommendation!
